In [1]:
import pandas as pd
import openai

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

Read the sampled dataset with Amazon Inventory Data

In [2]:
df_items = pd.read_json("../../../data/meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines=True)

In [3]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN


In [4]:
list(df_items["features"].items())[0]

(0,
 ['100% Acrylic',
  'Elastic closure',
  'Hand Wash Only',
  '【Upgraded Bluetooth Beanie】Bosttor upgraded Bluetooth wireless technology offer much stable and strong connection, support music and calling, easy and fast to pair with your devices. Maximum transmission distance up to 33 feet. Built-in Stereo Speakers & Mic, enhance your music listening experience.',
  '【Long Working Time Music Hat】Built with easy-access USB charging port, simply charge it via the included USD cable. It takes 1-2 hours to get full charged. The battery offers continuous working hours up to 10 hours. Perfect for camping, hiking, skiing, hunting, jogging, cycling, dog walking, auto repair, etc.',
  '【LED Headlight Light Up Your Way】Hands-free LED light is rechargeable and removable. You can remove it to charge by laptop, power bank, socket, car charger, etc. The 4 LED lights can light up to 30 feet away. It points directional light toward what your eyes are looking at. It has 3 brightness settings.',
  '【O

In [5]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51b7qcj8dZL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/71G+JK9lg1L._AC_UL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51JvxXFcBwL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51JvxXFcBwL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71vYog2F1LL._AC_UL1300_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51sKkBrOCiL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51sKkBrOCiL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71GowojGFfL._AC_UL1300_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51k6uDxkZpL._AC_SR38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51k6uDxkZpL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71ODAwq

Preprocessing Functions 

In [6]:
def preprocess_description(row):
    return f"{row['title']} {' '.join(row['features'])}"

In [7]:
def extract_first_large_image(row):
    return row["images"][0].get("large", "")

In [8]:
df_items["preprocessed_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [9]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,AMAZON FASHION,"Bosttor Bluetooth Beanie Hat with Light, Headl...",4.5,2342,"[100% Acrylic, Elastic closure, Hand Wash Only...",[],26.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Beanie Hat with Bluetooth Headphon...,Bosttor,"[Electronics, Headphones, Earbuds & Accessorie...","{'Department': 'unisex-adult', 'Date First Ava...",B0BH41HYFZ,NaN,NaN,NaN,"Bosttor Bluetooth Beanie Hat with Light, Headl...",https://m.media-amazon.com/images/I/51b7qcj8dZ...
1,All Electronics,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",4.2,148,[【USB or USB C to Ethernet Adapter】The Etherne...,[],14.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],Aceele,"[Electronics, Computers & Accessories, Network...",{'Product Dimensions': '3.54 x 0.98 x 0.71 inc...,B0BKPB2YQ9,NaN,NaN,NaN,"Aceele USB and USB C to Ethernet Adapter, 3.3f...",https://m.media-amazon.com/images/I/41WsGRr-3T...
2,All Electronics,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,4.2,3331,[📍3-Port HDMI Switch: This aluminum HDMI switc...,[],18.98,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Darren reviews VWRHAR 3-1 splitter...,VWRHar,"[Electronics, Home Audio, Home Audio Accessori...",{'Package Dimensions': '6.1 x 4.21 x 0.94 inch...,B09MM5QT3R,NaN,NaN,NaN,HDMI Switch 3 in 1 Out 4K UHD HDMI Switcher Sp...,https://m.media-amazon.com/images/I/41fsjaknf1...
3,All Electronics,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",3.4,125,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Burxoe,[],{'Package Dimensions': '3.15 x 3.07 x 2.52 inc...,B09Q5TNDHY,NaN,NaN,NaN,"Smart Watch,Ip67 Waterproof Bluetooth Smartwat...",https://m.media-amazon.com/images/I/51YFypeuZh...
4,Cell Phones & Accessories,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",4.7,335,[【Compatible Airpods 1/2 】This case designed f...,[1],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'airpod cases', 'url': 'https://www...",UGUHY,"[Electronics, Headphones, Earbuds & Accessorie...",{'Package Dimensions': '7.44 x 4.61 x 1.57 inc...,B0BZJKX7MS,NaN,NaN,NaN,"(3 Pack) Cute Airpod Case for Airpods 2&1,3D D...",https://m.media-amazon.com/images/I/41V7Wi2ckb...


Sample 50 items from dataset

In [10]:
df_sample = df_items.sample(n=50, random_state=42)

In [11]:
len(df_sample)

50

In [12]:
df_data_to_embed = df_sample[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]

In [13]:
df_data_to_embed.head()

,preprocessed_description,image,rating_number,price,average_rating,parent_asin
521,Marame 120mm 5v USB Powered Fan with Speed Con...,https://m.media-amazon.com/images/I/41Zpg4vSEj...,104,14.99,4.7,B0BRJS644Z
737,"Kids Wireless Headphones, Adjustable Headband,...",https://m.media-amazon.com/images/I/41+jaEbUbY...,173,NaN,4.2,B09KQP2H7N
740,"Veetone Aux Cord for iPhone, 3.3ft [Apple MFi ...",https://m.media-amazon.com/images/I/31ag2Blvjm...,2218,5.99,4.0,B0CC4HBS85
660,AICHESON Laptop Cooler Pad with 6 Cooling Fans...,https://m.media-amazon.com/images/I/51BKtk3-RD...,145,NaN,4.2,B099N9F3FP
411,"Raymate Bluetooth Speakers, HiFi Stereo Sound ...",https://m.media-amazon.com/images/I/31SnPdXS77...,270,38.69,4.7,B0C996WY16


In [14]:
data_to_embed = df_data_to_embed.to_dict(orient="records")

In [15]:
data_to_embed

[{'preprocessed_description': 'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sittin

Define the embedding function

In [16]:
response = openai.embeddings.create(
    input="Random text",
    model="text-embedding-3-small"
)

In [17]:
response

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.0159454345703125, -0.0039520263671875, 0.00843048095703125, 0.0036773681640625, -0.042877197265625, -0.062744140625, -0.0078887939453125, 0.011993408203125, -0.013092041015625, -0.0061187744140625, -0.006755828857421875, 0.0118255615234375, -0.020721435546875, 0.03778076171875, 0.0284423828125, -0.0185699462890625, -0.03204345703125, -0.006649017333984375, -0.05047607421875, 0.033660888671875, -0.0155181884765625, 0.024322509765625, -0.02020263671875, 5.1975250244140625e-05, 0.0191650390625, -0.0015325546264648438, 0.0158843994140625, -0.00872039794921875, 0.03643798828125, -0.036376953125, 0.0244598388671875, -0.04296875, 0.0224609375, -0.0254364013671875, 0.01399993896484375, 0.00911712646484375, 0.050384521484375, 0.0001552104949951172, 0.0225372314453125, 0.0023021697998046875, -0.0031719207763671875, -0.04742431640625, 0.031585693359375, 0.0275115966796875, 0.03009033203125, 0.026275634765625, -0.0198516845703125, -0.0075569152

In [18]:
len(response.data[0].embedding)

1536

In [25]:
def get_embedding(text,model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


In [26]:
get_embedding("Hi.")

[0.01244354248046875,
 -0.0029582977294921875,
 0.0164947509765625,
 0.007389068603515625,
 -0.01091766357421875,
 -0.00948333740234375,
 -0.033172607421875,
 0.04443359375,
 -0.01084136962890625,
 -0.048187255859375,
 -0.0018243789672851562,
 -0.033538818359375,
 -0.0014133453369140625,
 -0.027069091796875,
 0.034515380859375,
 0.027069091796875,
 -0.04180908203125,
 0.03704833984375,
 -0.034515380859375,
 0.057281494140625,
 0.05670166015625,
 0.015838623046875,
 -0.001789093017578125,
 0.016815185546875,
 0.002819061279296875,
 0.0049896240234375,
 0.0784912109375,
 -0.0032196044921875,
 0.0020618438720703125,
 -0.006679534912109375,
 0.03338623046875,
 -0.0382080078125,
 -0.0112457275390625,
 -0.0180206298828125,
 -0.00041365623474121094,
 -0.005603790283203125,
 -0.0195770263671875,
 0.0303802490234375,
 0.01499176025390625,
 -0.04833984375,
 0.032501220703125,
 -0.01004791259765625,
 -0.006053924560546875,
 0.0206146240234375,
 0.0013942718505859375,
 0.005702972412109375,
 -0.04

Create Qdrant Collection

In [27]:
qdrant_client = QdrantClient(url="http://localhost:6334")

/Users/gaurisharma/developer/ai-engineering-bootcamp/cohort_5/ai-engineering-bootcamp-amazon-assistant/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:295: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [28]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    on_disk_payload=True
)
qdrant_client.get_collection("Amazon-items-collection-01")


ResponseHandlingException: illegal request line